# Session 8

In [ ]:
pip install pygame opencv-python

In [8]:
import random
import pygame
try:
    from ugot import ugot
    got = ugot.UGOT()
    got.initialize("10.65.48.212") #192.168.88.1
    GYRO_AVAILABLE = True
except:
    got = None
    GYRO_AVAILABLE = False
pygame.init()
WIDTH, HEIGHT = 600, 400
screen = pygame.display.set_mode((WIDTH, HEIGHT))
font = pygame.font.SysFont(None, 32)

ROAD_WIDTH = 400
ROAD_LEFT = (WIDTH - ROAD_WIDTH) // 2
ROAD_RIGHT = ROAD_LEFT + ROAD_WIDTH
LANE_MARK_WIDTH = 8
LANE_MARK_HEIGHT = 40
LANE_MARK_SPACING = 30

CAR_WIDTH = 40
CAR_HEIGHT = 60
CAR_COLOR = (200, 30, 30)

OBSTACLE_WIDTH = 50
OBSTACLE_HEIGHT = 40
OBSTACLE_COLOR = (30, 30, 180)
OBSTACLE_SPEED = 5
SPAWN_INTERVAL = 900

MAX_X = ROAD_RIGHT - 20
MIN_X = ROAD_LEFT + 20
MAX_Y = HEIGHT - 20
MIN_Y = 20

def draw_road(scroll_offset):
    screen.fill((80, 170, 80)) # grass color
    pygame.draw.rect(screen, (40,40,40), (ROAD_LEFT,0,ROAD_WIDTH,HEIGHT))
    pygame.draw.rect(screen, (255,255,255), (ROAD_LEFT,0, 10,HEIGHT))
    pygame.draw.rect(screen, (255,255,255), (ROAD_RIGHT-10,0,10,HEIGHT))
    for y in range(-LANE_MARK_HEIGHT, HEIGHT, LANE_MARK_HEIGHT+LANE_MARK_SPACING):
        draw_y = y + scroll_offset % (LANE_MARK_HEIGHT+LANE_MARK_SPACING)
        pygame.draw.rect(screen,(255,255,255),
            (WIDTH//2-LANE_MARK_WIDTH//2, draw_y, 
            LANE_MARK_WIDTH,LANE_MARK_HEIGHT))

def draw_car(x, y):
    rect = pygame.Rect(int(x), int(y), CAR_WIDTH, CAR_HEIGHT)
    pygame.draw.rect(screen, CAR_COLOR, rect)

def draw_obstacle(obstacle):
    pygame.draw.rect(screen, OBSTACLE_COLOR, obstacle)
    pygame.draw.rect(screen, (255,255,255), obstacle, 2)

def clamp(value, low, high):
    return max(low, min(high, value))

x, y = WIDTH // 2, HEIGHT // 2
scroll_offset = 0
score = 0
level = 1
speed = 1
if GYRO_AVAILABLE:
    center_pitch, center_roll, center_yaw, _, _, _, _, _, _ = got.read_gyro_data()
else:
    center_pitch, center_roll, center_yaw = 0,0,0
last_spawn = pygame.time.get_ticks()
obstacles = []
game_over = False
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE and game_over:
                game_over = False
                x, y = WIDTH // 2, HEIGHT // 2
                scroll_offset = 0
                score = 0
                level = 1
                speed = 1
                obstacles.clear()
                last_spawn = pygame.time.get_ticks()
                OBSTACLE_SPEED = 5
                SPAWN_INTERVAL = 900
    
    if not game_over:
        # moved from the top of the while loop
        # -------------------------
        score += 1
        if score % 500 == 0:
            level += 1
            OBSTACLE_SPEED += 1
            SPAWN_INTERVAL -= 50
            SPAWN_INTERVAL = max(SPAWN_INTERVAL, 200)
            speed += 1
        # -------------------------
        if GYRO_AVAILABLE:
            pitch, roll, yaw, _, _, _, _, _, _ = got.read_gyro_data()
        else:
            pitch, roll, yaw = 0,0,0
        move_x = (roll - center_roll) / 20
        move_y = (pitch - center_pitch) / -20

        # keyboard control
        keys = pygame.key.get_pressed()
        if keys[pygame.K_LEFT]:
            move_x = -2
        if keys[pygame.K_RIGHT]:
            move_x = 2
        if keys[pygame.K_UP]:
            move_y = -2
        if keys[pygame.K_DOWN]:
            move_y = 2

        x += move_x * speed
        y += move_y * speed
        x = clamp(x, MIN_X, MAX_X)
        y = clamp(y, MIN_Y, MAX_Y)

        now = pygame.time.get_ticks()
        if now - last_spawn > SPAWN_INTERVAL:
            last_spawn = now
            obs_x = random.randint(ROAD_LEFT, ROAD_RIGHT)
            obstacles.append(pygame.Rect(obs_x, -OBSTACLE_HEIGHT, 
                    OBSTACLE_WIDTH, OBSTACLE_HEIGHT))

        for obs in obstacles:
            obs.y += OBSTACLE_SPEED
        
        obstacles = [obs for obs in obstacles if obs.y<HEIGHT+OBSTACLE_HEIGHT]

        player_rect = pygame.Rect(int(x), int(y), CAR_WIDTH, CAR_HEIGHT)
        if any(player_rect.colliderect(obs) for obs in obstacles):
            print(f"Game over! Your score is {score}")
            # running = False
            game_over = True

    # screen.fill((255, 255, 255))
    scroll_offset += 7
    draw_road(scroll_offset)
    for obs in obstacles:
        draw_obstacle(obs)
    # pygame.draw.circle(screen, (0,255,0), (int(x), int(y)), 20)
    draw_car(x, y)
    score_text = font.render(f"Score: {score}", True, (255,255,0))
    screen.blit(score_text, (20, 20))
    level_text = font.render(f"Level: {level}", True, (255,255,0))
    screen.blit(level_text, (20, 50))
    if game_over:
        game_over_text = font.render(f"Game over! SPACE to restart", True, (255,255,0))
        screen.blit(game_over_text, (20, 80))
    pygame.display.flip()

pygame.quit()

10.65.48.212:50051
Game over! Your score is 218


In [12]:
import cv2
import numpy as np
from ugot import ugot
got = ugot.UGOT()
got.initialize("10.65.48.212") # 192.168.88.1
got.open_camera()
got.balance_start_balancing()

import pygame
pygame.init()
screen = pygame.display.set_mode((640, 480))
pygame.display.set_caption("UGOT camera")

move_speed = 30
turn_speed = 45

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    frame = got.read_camera_data()
    if not frame:
        continue
    nparr = np.frombuffer(frame, np.uint8)
    data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    flipped = cv2.flip(data, 1)
    frame_rgb = cv2.cvtColor(flipped, cv2.COLOR_BGR2RGB)
    surface = pygame.image.frombuffer(frame_rgb.tobytes(), 
                (640,480), "RGB")
    screen.blit(surface, (0,0))

    keys = pygame.key.get_pressed()
    direction = 0
    move_speed = 30
    if keys[pygame.K_w]:
        direction = 0
    elif keys[pygame.K_s]:
        direction = 1
    else:
        move_speed = 0
    turn = 2
    turn_speed = 45
    if keys[pygame.K_a]:
        turn = 2 # left
    elif keys[pygame.K_d]:
        turn = 3 # right
    else:
        turn_speed = 0
    got.balance_move_turn(direction, move_speed, turn, turn_speed)

    pygame.display.flip()
pygame.quit()

10.65.48.212:50051
